# 在 Google Colab 上跑 gpt-oss-20b

> 原文：[articles/gpt-oss/run-colab.ipynb](https://github.com/openai/openai-cookbook/blob/main/articles/gpt-oss/run-colab.ipynb)



## 這篇在做什麼
示範在免費的 Google Colab 上用 Transformers 跑 gpt-oss-20b。因為模型原生就是 MXFP4 量化，資源有限的環境也跑得動。內容包含裝套件、載入模型、組 messages，以及調整 reasoning effort。

## 學完你會
- 在 Colab 裝好支援 MXFP4 的 PyTorch、Triton kernels 和最新版 transformers。
- 載入 `openai/gpt-oss-20b`，用 chat template 組 messages 並產生回應。
- 用 `apply_chat_template()` 的參數把 reasoning effort 設成 `low`、`medium` 或 `high`。

## 開跑前
- 需要 Colab 的 GPU runtime（免費版的 T4 就可以）。
- 裝完套件要重啟 runtime。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U transformers
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set articles/gpt-oss
os.chdir('openai-cookbook/articles/gpt-oss')

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openai/openai-cookbook/blob/main/articles/gpt-oss/run-colab.ipynb)

# 在免費的 Google Colab 上跑 OpenAI gpt-oss 20B

OpenAI 釋出了 `gpt-oss` [120B](https://hf.co/openai/gpt-oss-120b) 和 [20B](https://hf.co/openai/gpt-oss-20b)，兩個模型都是 Apache 2.0 授權。

其中 `gpt-oss-20b` 是為了低延遲和本機或特定用途設計的（21B 參數，其中 3.6B 是 active 參數）。

因為模型是以原生 MXFP4 量化訓練的，連 Google Colab 這種資源有限的環境也能輕鬆跑 20B。

作者：[Pedro](https://huggingface.co/pcuenq) 和 [VB](https://huggingface.co/reach-vb)

## 設定環境

因為 transformers 對 mxfp4 的支援還很新，需要較新版本的 PyTorch 和 CUDA，才能安裝 `mxfp4` 的 triton kernels。

我們也要從原始碼安裝 transformers，並解除安裝 `torchvision` 和 `torchaudio` 來避免相依套件衝突。

In [ ]:
!pip install -q --upgrade torch

In [ ]:
!pip install -q transformers triton==3.4 kernels

In [ ]:
!pip uninstall -q torchvision torchaudio -y

裝完上面的套件後，請重啟 Colab 的 runtime session。

## 在 Google Colab 從 Hugging Face 載入模型

模型從這裡載入：[openai/gpt-oss-20b](https://hf.co/openai/gpt-oss-20b)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="cuda",
)

## 設定 messages / 對話

你可以提供選用的 system prompt，或直接給輸入。

In [ ]:
messages = [
    {"role": "system", "content": "Always respond in riddles"},
    {"role": "user", "content": "What is the weather like in Madrid?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))

## 指定 Reasoning Effort

把它當作額外參數傳給 `apply_chat_template()` 就好。支援的值有 `"low"`、`"medium"`（預設）或 `"high"`。

In [ ]:
messages = [
    {"role": "system", "content": "Always respond in riddles"},
    {"role": "user", "content": "Explain why the meaning of life is 42"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="high",
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=500)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))

## 試試其他 prompt 和點子

更多點子可以看我們的 blog 文章：[https://hf.co/blog/welcome-openai-gpt-oss](https://hf.co/blog/welcome-openai-gpt-oss)

## 重點整理
- gpt-oss-20b 原生 MXFP4 量化，所以免費 Colab 也跑得動。
- MXFP4 支援還很新，要用新版 PyTorch、CUDA 和從原始碼裝的 transformers，裝完要重啟 runtime。
- Messages 靠 chat template 轉成模型輸入，system prompt 是選用的。
- Reasoning effort 是 chat template 的參數，分 `low`、`medium`、`high`，越高通常想得越久。

## 練習
1. 把 messages 裡的問題換成你自己的，並加上一個 system prompt 指定回答風格。
2. 把 reasoning effort 從 `medium` 改成 `low` 和 `high`，比較回應長度和花費時間。
3. 用同一個 prompt 在三種 effort 下各跑一次，把輸出的 reasoning 部分和最終答案分開印出來比較。